# Laboratorio de rendimiento

Este notebook está dividido en dos partes:

1. **Python puro**: para estudiar complejidad algorítmica, overhead y casos CPU-bound sin ruido de red, HTTP o base de datos.
2. **Servicios de la API**: para medir la lógica real del proyecto y discutir cómo conviven complejidad, ORM, transacciones y operaciones I/O-bound.

## Parte 2: Servicios de la API

En esta sección medimos la lógica del proyecto sin pasar por HTTP. La idea es evitar ruido del framework, pero conservar el costo real del ORM, la base de datos y las transacciones.

Acá conviene agregar dos preguntas más:

1. ¿La operación es principalmente CPU-bound o I/O-bound?
2. ¿El cuello de botella está en Python o en la base de datos?

In [ ]:
await create_db_and_tables()
print("Base de datos lista.")

In [ ]:
async def clear_tables(*models):
    async with async_session_factory() as session:
        for model in models:
            await session.exec(delete(model))
        await session.commit()


async def seed_persons(quantity: int) -> None:
    await clear_tables(Person)
    people = [
        Person(name=f"person_{i}", passport=f"P{i:06d}", age=i % 100)
        for i in range(quantity)
    ]
    async with async_session_factory() as session:
        session.add_all(people)
        await session.commit()


async def seed_inventory(quantity: int, matching_every: int = 10) -> None:
    await clear_tables(Inventory)
    items = [
        Inventory(
            name="producto_1" if i % matching_every == 0 else f"producto_{i}",
            amount=i % 250,
        )
        for i in range(quantity)
    ]
    async with async_session_factory() as session:
        session.add_all(items)
        await session.commit()


async def count_rows(model) -> int:
    async with async_session_factory() as session:
        rows = await session.exec(select(model))
        return len(rows.all())

### Ejemplo C: múltiples `commit()` vs un commit por lote

Los dos caminos son lineales en `n`, pero su rendimiento práctico puede diferir muchísimo. Este ejemplo es ideal para mostrar una operación donde el costo dominante suele ser **I/O-bound**.

In [ ]:
async def run_user_bulk_for(quantity: int):
    await clear_tables(User)
    async with async_session_factory() as session:
        service = UserService(session)
        return await service.create_users_for(quantity)


async def run_user_bulk_numpy(quantity: int):
    await clear_tables(User)
    async with async_session_factory() as session:
        service = UserService(session)
        return await service.create_users_numpy(quantity)

In [ ]:
quantity = 500

report_for = await benchmark_async(lambda: run_user_bulk_for(quantity), repeats=3)
report_numpy = await benchmark_async(lambda: run_user_bulk_numpy(quantity), repeats=3)

resumir("bulk_for", report_for)
resumir("bulk_numpy", report_numpy)

In [ ]:
_result, _stats = await profile_async(lambda: run_user_bulk_for(200), limit=25)

### Ejemplo D: `add()` en bucle vs `add_all()`

Ambos caminos siguen siendo `O(n)`, pero el perfil sirve para ver dónde aparece el costo real: construcción de objetos, operaciones del ORM y persistencia.

In [ ]:
async def run_inventory_create_all(quantity: int):
    await clear_tables(Inventory)
    inventories = [Inventory(name=f"producto_{i}", amount=i) for i in range(quantity)]
    async with async_session_factory() as session:
        service = InventoryService(session)
        return await service.create_all(inventories)


async def run_inventory_create_all_async(quantity: int):
    await clear_tables(Inventory)
    inventories = [Inventory(name=f"producto_{i}", amount=i) for i in range(quantity)]
    async with async_session_factory() as session:
        service = InventoryService(session)
        return await service.create_all_async(inventories)

In [ ]:
quantity = 2_000

report_add = await benchmark_async(lambda: run_inventory_create_all(quantity), repeats=3)
report_add_all = await benchmark_async(lambda: run_inventory_create_all_async(quantity), repeats=3)

resumir("inventory_create_all", report_add)
resumir("inventory_create_all_async", report_add_all)

### Ejemplo E: trabajo en memoria después de leer desde la base

Estos casos mezclan fases distintas:

- traer filas desde la base de datos
- materializar objetos del ORM
- filtrar u ordenar en Python

Eso permite discutir qué parte es I/O-bound y cuál pasa a ser CPU-bound.

In [ ]:
await seed_persons(10_000)

async with async_session_factory() as session:
    person_service = PersonService(session)
    report_people = await benchmark_async(person_service.get_persons_sort, repeats=3)

resumir("persons_sort", report_people)
print("filas_person", await count_rows(Person))

In [ ]:
await seed_inventory(15_000, matching_every=7)

async with async_session_factory() as session:
    inventory_service = InventoryService(session)
    report_inventory_python = await benchmark_async(inventory_service.get_inventory, repeats=3)
    report_inventory_db = await benchmark_async(inventory_service.get_inventory_all, repeats=3)

resumir("inventory_python_filter_sort", report_inventory_python)
resumir("inventory_db_where_order", report_inventory_db)

## Preguntas guía

### Para Python puro

1. ¿Qué implementación tiene mejor complejidad asintótica?
2. ¿En qué rango de `n` la implementación teóricamente peor sigue siendo competitiva?
3. ¿Qué muestra el profiler sobre el trabajo en CPU?

### Para servicios de la API

1. ¿La diferencia de rendimiento viene de la complejidad o del patrón de acceso a la base?
2. ¿Dónde aparece el costo dominante: Python, ORM o transacciones?
3. ¿Qué ejemplo es principalmente CPU-bound y cuál es principalmente I/O-bound?
4. ¿Qué optimización cambia el algoritmo y cuál solo reduce overhead?